# Modules

In [ ]:
from collections import defaultdict
from concurrent import futures
import json
import numpy as np
import os
import pandas as pd

pd.set_option('display.max_columns', 500)

# sys.path.append(str(Path("src/").resolve()))

from longitudinal_arm_swing.constants import *
from longitudinal_arm_swing.utils import defaultdict_to_dict, extract_med_info, get_med_ids, \
    get_no_med_ids, get_start_med_ids, add_updrs_columns, set_data_types, \
        determine_affected_side_watch_side_mapping, generate_ids_bins, determine_excluded_ids_med_split, \
        strip_med_suffix_id

# Functions

In [2]:
def load_json_file(filepath):
    subject = os.path.splitext(os.path.basename(filepath))[0]
    with open(filepath, 'r') as f:
        return subject, json.load(f)

# Constants

In [3]:
start_week_y1 = 2
start_week_y2 = 52

end_week_y1 = 50
end_week_y2 = 100

max_week_gap = 4  # Maximum gap (in weeks) allowed between two consecutive weeks with sufficient data

min_hours_per_day_sufficient_sensor_data = 10  # minimum number of hours of sensor data required for a day to be valid
min_valid_days_per_week_sufficient_sensor_data = 3  # minimum number of valid days (based on sensor data) required for a week to be valid

min_minutes_per_day_arm_swing = 2  # minimum number of minutes of arm swing data required for a day to be valid
min_valid_days_per_week_arm_swing = 2  # minimum number of valid days (based on arm swing data) required for a week to be valid

min_valid_weeks_l1tf = 8  # Minimum number of valid weeks required for l1 trend filtering

datasets = ['ppp', 'denovo', 'controls']

pdq_cat_cols = [
    'pdq39_mobility', 'pdq39_adl', 'pdq39_emotional_wellbeing', 'pdq39_stigma', 'pdq39_social_support', 
    'pdq39_cognition', 'pdq39_communication', 'pdq39_bodily_discomfort', 'pdq39_total'
]

# Load data

In [ ]:
digital_measures = defaultdict(lambda: defaultdict(dict))

for dataset in datasets:
    for week_nr in WEEKS_PER_DATASET[dataset]:
        aggregation_week_path = PATHS_PER_DATASET[dataset]['aggregations']
        week_path = aggregation_week_path / str(week_nr)

        json_files = [week_path / f for f in os.listdir(week_path) if f.endswith('.json')]

        with futures.ThreadPoolExecutor() as executor:
            results = executor.map(load_json_file, json_files)

        for subject, data in results:
            digital_measures[dataset][week_nr][subject] = data

digital_measures = defaultdict_to_dict(digital_measures)

# IDs processed per step
step_ids = {}
with open(os.path.join(PATH_IDS, PROCESSED_IDS_FILENAME), 'r') as f:
    step_ids = json.load(f)

# Statistics of processed data
stats = {}
for dataset in datasets:
    with open(PATH_STATS / STATS_FILENAMES_PER_DATASET[dataset], 'r') as f:
        stats[dataset] = json.load(f)

# Per participant the weeks of their visits
with open(PATH_CLINICAL_DATA / 'visit_weeks.json', 'r') as f:
    visit_weeks = json.load(f)

# Per participants if and when they started dopaminergic medication
start_med_week_dict = {
    'ppp': pd.read_csv(PATH_CLINICAL_DATA / 'ppp' / PPP_START_MED_FILENAME),
    'denovo': pd.read_csv(PATH_CLINICAL_DATA / 'denovo' / DENOVO_START_MED_WEEK_FILENAME)
}

In [ ]:
# Set maximum allowed deviation from the final measurement week
visit_nr = 3
margin = 20
ids_late_visit_3 = []

for dataset in ['ppp', 'denovo']:
    ids_late_visit_3 += [
        x for x in visit_weeks[dataset] 
        if visit_weeks[dataset][x][f'visit{visit_nr}'] is not None 
        and not pd.isna(visit_weeks[dataset][x][f'visit{visit_nr}']) 
        and visit_weeks[dataset][x][f'visit{visit_nr}'] != 'null'
        and (
            int(visit_weeks[dataset][x][f'visit{visit_nr}']) > end_week_y2 + margin
            or int(visit_weeks[dataset][x][f'visit{visit_nr}']) < end_week_y2 - margin
        )
    ]

with open(PATH_IDS / 'ids_late_visit_3.txt', 'w') as f:
    for id in ids_late_visit_3:
        f.write(f'{id}\n')

In [ ]:
clinical_data = defaultdict(lambda: defaultdict(dict))
for dataset in DATASETS:
    for visit_nr, visit_filename in VISIT_FILENAMES_PER_DATASET[dataset].items():
        clinical_data[dataset]['visits'][visit_nr] = pd.read_csv(PATHS_PER_DATASET[dataset]['clinical'] / visit_filename)
        
        if dataset == 'ppp':
            clinical_data[dataset]['visits'][visit_nr]['id'] = clinical_data[dataset]['visits'][visit_nr]['id'].apply(
                lambda x: f'POMU{x[:16]}'
            )
        elif dataset == 'controls':
            clinical_data[dataset]['visits'][visit_nr] = clinical_data[dataset]['visits'][visit_nr].rename(
                columns={'AssDatYear': 'AssessYear', 'AssDatMonth': 'AssessMonth'}
            )

clinical_data['ppp']['ledd'] = pd.read_csv(
    PATHS_PER_DATASET['ppp']['clinical'] / LEDD_PPP_FILENAME
)

clinical_data = defaultdict_to_dict(clinical_data)

## Prepare data

In [7]:
# Determine whether participants have sufficient sensor data
sufficient_sensor_data = defaultdict(lambda: defaultdict(dict))
for dataset in datasets:
    for week, week_data in stats[dataset].items():
        sufficient_sensor_data[dataset][int(week)] = {
            subject: sum(
                week_data[subject]['hours_data'][weekday]['hours_between_8_and_22'] >= min_hours_per_day_sufficient_sensor_data
                for weekday in week_data[subject]['hours_data']
            ) >= min_valid_days_per_week_sufficient_sensor_data
            for subject in week_data.keys()
        }

sufficient_sensor_data = defaultdict_to_dict(sufficient_sensor_data)

# Prepare medication info
clinical_data['ppp']['visits'][3].loc[clinical_data['ppp']['visits'][3]['id'] == 'POMU30163E78B5A9CCAB', 'ParkinMedUser'] = 0  # Starts medication AFTER study
clinical_data['ppp']['visits'][2].loc[clinical_data['ppp']['visits'][2]['id'] == 'POMU40FDD170B7D6C89E', 'ParkinMedUser'] = 0  # Does not start medication
clinical_data['ppp']['visits'][2].loc[clinical_data['ppp']['visits'][2]['id'] == 'POMU42A89EF7BECED9AD', 'ParkinMedUser'] = 0  # Does not start medication

med_info_ids = {'visits': {}, 'groups': {}}
denovo_med_colnames = {1: 'ParkinMedUser', 2: 'Up3OfParkMedic', 3: 'Up3OfParkMedic'}

# Extract medication status per visit for each dataset
for visit in [1, 2, 3]:
    med_info_ids['visits'][visit] = {
        'ppp': extract_med_info(clinical_data['ppp']['visits'][visit], 'ParkinMedUser'),
        'denovo': extract_med_info(clinical_data['denovo']['visits'][visit], denovo_med_colnames[visit])
    }

# For each dataset, assign participants to groups based on medication status
med_info_ids['groups'] = {
    'med': {group: get_med_ids(med_info_ids, group) for group in PD_DATASETS},
    'no_med': {group: get_no_med_ids(med_info_ids, group, start_med_week_dict[group]) for group in PD_DATASETS},
    'start_med': {group: get_start_med_ids(med_info_ids, group, start_med_week_dict[group]) for group in PD_DATASETS},
}

# Move participants who start medication after study to the no_med group
for dataset in PD_DATASETS:
    start_med_ids_dataset = start_med_week_dict[dataset]['ID'].values
    for id in start_med_ids_dataset:
        final_visit_week = visit_weeks[dataset][id]["visit3"]
        if final_visit_week is None:
            final_visit_week = 104
        start_med_week = start_med_week_dict[dataset].loc[start_med_week_dict[dataset]['ID'] == id, 'StartWeek'].values[0]
        if start_med_week >= min(final_visit_week, 104):
            start_med_week_dict[dataset] = start_med_week_dict[dataset][start_med_week_dict[dataset]['ID'] != id]
            if id in med_info_ids['groups']['start_med'][dataset]:
                med_info_ids['groups']['start_med'][dataset].remove(id)
            if id not in med_info_ids['groups']['no_med'][dataset]:
                med_info_ids['groups']['no_med'][dataset].append(id)

In [8]:
dataset_ids = {}
affected_side_ids = {}
watch_side_dict = defaultdict(lambda: defaultdict(dict))
final_dfs = []

for dataset in datasets:
    # Load IDs
    dataset_ids[dataset] = list(pd.concat(
        [clinical_data[dataset]['visits'][visit_nr]['id'] for visit_nr in VISITS_PER_DATASET[dataset]]
    ).unique())

    # Set data types
    for visit_nr in VISITS_PER_DATASET[dataset]:
        df_visit = clinical_data[dataset]['visits'][visit_nr] 

        if dataset != 'controls':
            if visit_nr == 1:
                numeric_cols = UPDRS_COLS_PER_DATASET[dataset] + NUMERIC_COLS_VISIT_1_PER_DATASET[dataset]
            else:
                numeric_cols = UPDRS_COLS_PER_DATASET[dataset] + NUMERIC_COLS_VISITS

            df_visit = set_data_types(df_visit, numeric_cols)

        # Apply mappings
        df_visit['WatchSide'] = df_visit['WatchSide'].apply(pd.to_numeric, errors='coerce').apply(lambda x: WATCH_SIDE_MAPPING[x] if pd.notna(x) else np.nan)
        df_visit.loc[:, 'PrefHand'] = df_visit['PrefHand'].apply(pd.to_numeric, errors='coerce')

        if dataset != 'controls' and visit_nr == 1:
            df_visit['MostAffSide'] = df_visit['MostAffSide'].apply(pd.to_numeric, errors='coerce').map(AFFECTED_SIDE_MAPPING)      

        # Determine the watch side of participants per visit
        for subject in dataset_ids[dataset]:
            df_subject = df_visit.loc[df_visit['id'] == subject]
        
            if df_subject.shape[0] > 0:
                watch_side_dict[dataset][subject][str(visit_nr)] = df_subject['WatchSide'].values[0]
            else:
                watch_side_dict[dataset][subject][str(visit_nr)] = np.nan

        clinical_data[dataset]['visits'][visit_nr] = df_visit

    # Years since diagnosis (float)
    if dataset == 'ppp':
        clinical_data['ppp']['visits'][1]['YearsSinceDiagFloat'] = clinical_data['ppp']['visits'][1]['MonthSinceDiag'] / 12
    elif dataset == 'denovo':
        clinical_data['denovo']['visits'][1].loc[:, 'YearsSinceDiagFloat'] = clinical_data['denovo']['visits'][1].apply(
            lambda x: (x['AssessYear'] + x['AssessMonth'] / 12) -  (x['DiagParkYear'] + x['DiagParkMonth'] / 12), axis=1 
        )

    # Determine the watch side of participants based on the three visits
    for subject, watch_side_vals in watch_side_dict[dataset].items():
        # Option 1: all three are the same and either 'left' or 'right'
        if len(set(watch_side_vals.values())) == 1 and watch_side_vals['1'] in ['left', 'right']:
            watch_side_dict[dataset][subject]['final'] = watch_side_vals['1']
        # Option 2: all three are the same but all nan
        elif all(pd.isna(val) for val in watch_side_vals.values()):
            watch_side_dict[dataset][subject]['final'] = 'unknown'
        # Option 3: excluding nans, all are the same
        elif len(set([val for val in watch_side_vals.values() if pd.notna(val)])) == 1:
            watch_side_dict[dataset][subject]['final'] = [val for val in watch_side_vals.values() if pd.notna(val)][0]
        # Option 4: excluding nans, there are two values
        elif len(set([val for val in watch_side_vals.values() if pd.notna(val)])) == 2:
            watch_side_dict[dataset][subject]['final'] = 'switched'

    with open(os.path.join(PATH_CLINICAL_DATA, 'watch_side_dict.json'), 'w') as f:
        json.dump(watch_side_dict, f, indent=2)

    for visit_nr in VISITS_PER_DATASET[dataset]:
        df_visit = clinical_data[dataset]['visits'][visit_nr] 

        # Change watch side to the final value if it is not 'switched'
        df_visit['WatchSide'] = df_visit['id'].apply(
            lambda x: watch_side_dict[dataset][x]['final'] 
            if watch_side_dict[dataset][x]['final'] != 'switched'
        else watch_side_dict[dataset][x][str(visit_nr)]
        )

        # Create UPDRS columns
        if dataset in PD_DATASETS:
            df_visit = add_updrs_columns(df_visit, dataset)
            df_visit['pdq39_mobility'] = df_visit[PDQ_MOBILITY_COLS].sum(axis=1)
            df_visit['pdq39_adl'] = df_visit[PDQ_ADL_COLS].sum(axis=1)
            df_visit['pdq39_emotional_wellbeing'] = df_visit[PDQ_EMO_COLS].sum(axis=1)
            df_visit['pdq39_stigma'] = df_visit[PDQ_STIGMA_COLS].sum(axis=1)
            df_visit['pdq39_social_support'] = df_visit[PDQ_SOCSUP_COLS].sum(axis=1)
            df_visit['pdq39_cognition'] = df_visit[PDQ_COG_COLS].sum(axis=1)
            df_visit['pdq39_communication'] = df_visit[PDQ_COMM_COLS].sum(axis=1)
            df_visit['pdq39_bodily_discomfort'] = df_visit[PDQ_BODDIS_COLS].sum(axis=1)
            df_visit['pdq39_total'] = df_visit[PDQ_ALL_COLS].sum(axis=1)

            df_visit['rbd_subjective'] = df_visit[[
                'RemSbdq01', 'RemSbdq02', 'RemSbdq03', 'RemSbdq04', 'RemSbdq05', 'RemSbdq06', 'RemSbdq07', 
                'RemSbdq08', 'RemSbdq09', 'RemSbdq10', 'RemSbdq11', 'RemSbdq12'
            ]].sum(axis=1) + 1

        clinical_data[dataset]['visits'][visit_nr] = df_visit

    # Determine if watch is worn on most affected side
    if dataset in PD_DATASETS:
        visit_1_df = clinical_data[dataset]['visits'][1]
        affected_side_ids[dataset] = determine_affected_side_watch_side_mapping(visit_1_df, score_category='hypokinesia')

    for visit_nr in VISITS_PER_DATASET[dataset]:
        df_visit = clinical_data[dataset]['visits'][visit_nr].copy()

        cols_visit_dataset = [col for col in STORE_CLINICAL_COLS_PER_DATASET[dataset][visit_nr] if col in df_visit.columns]
        if dataset in PD_DATASETS:
            cols_visit_dataset.extend([col for col in df_visit.columns if col.startswith('updrs_')] + pdq_cat_cols + ['rbd_subjective'])

        df_visit = df_visit[cols_visit_dataset]

        df_visit.loc[:, 'dataset'] = dataset
        df_visit.loc[:, 'visit'] = visit_nr

        if dataset == 'ppp':
            # Add LEDD to ppp data
            ledd_df = clinical_data[dataset]['ledd'][['id', f'Visit{visit_nr}', f'Visit{visit_nr}_missing_med']]
            ledd_df.columns = ['id', 'ledd', 'missing_med']
            df_visit = df_visit.merge(ledd_df, on='id', how='left') 

        final_dfs.append(df_visit)

with open(os.path.join(PATH_IDS, 'affected_side_ids.json'), 'w') as f:
    json.dump(affected_side_ids, f, indent=2)

with open(os.path.join(PATH_IDS, 'dataset_ids.json'), 'w') as f:
    json.dump(dataset_ids, f, indent=2)

df_clinical = pd.concat(final_dfs, ignore_index=True).reset_index(drop=True)

for dataset in [ds for ds in datasets if ds in PD_DATASETS]:
    df_clinical.loc[df_clinical['dataset'] == dataset, 'affected_side'] = df_clinical.loc[df_clinical['dataset'] == dataset, 'id'].apply(
        lambda x: 'mas' if x in affected_side_ids[dataset]['mas'] else 'las' if x in affected_side_ids[dataset]['las'] else 'equal'
    )

# Participants using a walking aid
df_clinical['walking_aid'] = df_clinical['Updrs2It25'] >= 2

# Participants with significant dyskinesia
df_clinical['at_least_slight_dyskinesia'] = df_clinical['MotComDysKinTime'].isin(AT_LEAST_SLIGHT_DYSKINESIA_VALS)
df_clinical['at_least_significant_dyskinesia'] = df_clinical['MotComDysKinTime'].isin(AT_LEAST_SIGNIFICANT_DYSKINESIA_VALS)

# Save data
df_clinical.to_parquet(os.path.join(PATH_CLINICAL_DATA, 'clinical_data.parquet'))

# Prepare digital measures
dysk_start_exclude = 'significant'  # slight or significant

valid_arm_swing_days = {}

# Should have at least 2 minutes of arm swing data per day
for dataset in datasets:
    valid_arm_swing_days[dataset] = {}
    for week in digital_measures[dataset]:
        valid_arm_swing_days[dataset][week] = {}
        for subject in digital_measures[dataset][week].keys():
            if '0_inf' in digital_measures[dataset][week][subject]['filtered']['minutes_of_data']:
                n_valid_days = len([day for day, n_minutes in digital_measures[dataset][week][subject]['filtered']['minutes_of_data']['0_inf'].items() if n_minutes >= min_minutes_per_day_arm_swing])
                valid_arm_swing_days[dataset][week][subject] = n_valid_days

store_ids_by_category = {step: {dataset: {} for dataset in DATASETS} for step in ANALYSIS_STEPS}
ids_srm_weeks = {}

C:\Users\z665206\AppData\Local\Temp\ipykernel_23028\1419098055.py:44: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  clinical_data['ppp']['visits'][1]['YearsSinceDiagFloat'] = clinical_data['ppp']['visits'][1]['MonthSinceDiag'] / 12


In [9]:
start_week_threshold = 26
remaining_ids_per_dataset = {}
start_weeks_las = [2, 4, 6]
final_weeks_las = [96, 98, 100, 102, 104]

for dataset in [ds for ds in datasets if ds in ['ppp', 'denovo']]:
    print(f"Dataset: {dataset}\n--------------\n")
    las_ids = affected_side_ids[dataset]['las']

    no_med_during_study_las_ids = [id for id in med_info_ids['groups']['no_med'][dataset] if id in las_ids]
    no_med_alt_diagnosis_ids = [id for id in IDS_ALTERNATIVE_DIAGNOSIS if id in no_med_during_study_las_ids]
    no_med_missing_med_info_ids = [id for id in IDS_MED_INFO_PARTICIPANT_MISSING if id in no_med_during_study_las_ids]

    start_med_las_ids = [id for id in med_info_ids['groups']['start_med'][dataset] if id in las_ids]
    start_med_time_unknown_ids = [id for id in start_med_las_ids if id in IDS_START_MED_TIME_UNKNOWN]
    start_med_early_ids = list(start_med_week_dict[dataset].loc[start_med_week_dict[dataset]['StartWeek'] < start_week_threshold, 'ID'].values)
    start_weeks = list(start_med_week_dict[dataset].loc[start_med_week_dict[dataset]['ID'].isin(start_med_las_ids), 'StartWeek'].values)
    start_med_alt_diagnosis_ids = [id for id in IDS_ALTERNATIVE_DIAGNOSIS if id in start_med_las_ids]
    start_med_missing_med_info_ids = [id for id in IDS_MED_INFO_PARTICIPANT_MISSING if id in start_med_las_ids]

    print(f"Number of IDs LAS: {len(las_ids)} (out of {len(affected_side_ids[dataset]['las'] + affected_side_ids[dataset]['mas'])})")
    print(f"... of which no medication during entire study: {len(no_med_during_study_las_ids)}")
    print(f"    ... of which alternative diagnosis: {len(no_med_alt_diagnosis_ids)}")
    print(f"    ... of which missing medication info: {len(no_med_missing_med_info_ids)}")

    print(f"... of which start medication during study: {len(start_med_las_ids)}")
    print(f"    ... of which start med time unknown: {len(start_med_time_unknown_ids)}")
    print(f"    ... of which start medication after week {start_week_threshold}: {len([week for week in start_weeks if week >= start_week_threshold])}")
    print(f"    ... of which alternative diagnosis during study: {len(start_med_alt_diagnosis_ids)}")
    print(f"    ... of which missing medication info: {len(start_med_missing_med_info_ids)}")

    remaining_las_ids = [id for id in no_med_during_study_las_ids + start_med_las_ids if id not in start_med_time_unknown_ids + start_med_early_ids + no_med_alt_diagnosis_ids + start_med_alt_diagnosis_ids]

    print(f"\nNumber of IDs remaining after clinical exclusions: {len(remaining_las_ids)}\n")

    ids_excluded_measurements = {
        'start_weeks': [],
        'final_weeks': [],
    }

    for subject in remaining_las_ids:
        start_weeks_remaining = start_weeks_las.copy()
        final_weeks_remaining = final_weeks_las.copy()

        for week in start_weeks_las:
            if subject not in sufficient_sensor_data[dataset][week]:
                start_weeks_remaining.remove(week)
            elif sufficient_sensor_data[dataset][week][subject] == False:
                start_weeks_remaining.remove(week)
            elif subject not in valid_arm_swing_days[dataset][week]:
                start_weeks_remaining.remove(week)
            elif valid_arm_swing_days[dataset][week][subject] < min_valid_days_per_week_arm_swing:
                start_weeks_remaining.remove(week)
            
        if len(start_weeks_remaining) == 0:
            ids_excluded_measurements['start_weeks'].append(subject)

        for week in final_weeks_las:
            if subject not in digital_measures[dataset][week]:
                final_weeks_remaining.remove(week)
            elif subject not in sufficient_sensor_data[dataset][week]:
                final_weeks_remaining.remove(week)
            elif sufficient_sensor_data[dataset][week][subject] == False:
                final_weeks_remaining.remove(week)
            elif subject not in valid_arm_swing_days[dataset][week]:
                final_weeks_remaining.remove(week)
            elif valid_arm_swing_days[dataset][week][subject] < min_valid_days_per_week_arm_swing:
                final_weeks_remaining.remove(week)

        if len(final_weeks_remaining) == 0:
            ids_excluded_measurements['final_weeks'].append(subject)

    print(f"Number of IDs excluded for measurement reasons: {len(set(ids_excluded_measurements['start_weeks'] + ids_excluded_measurements['final_weeks']))}")
    remaining_las_ids = [subject for subject in remaining_las_ids if subject not in set(ids_excluded_measurements['start_weeks'] + ids_excluded_measurements['final_weeks'])]
    
    print(f"Number of IDs remaining after arm swing exclusions: {len(remaining_las_ids)}")

    remaining_las_ids_start_med = [subject for subject in remaining_las_ids if subject in start_med_las_ids]
    print(f"... Of which start with med (with start med weeks): {len(remaining_las_ids_start_med)}  {start_med_week_dict[dataset].loc[start_med_week_dict[dataset]['ID'].isin(remaining_las_ids_start_med), 'StartWeek'].values} \n")

    remaining_ids_per_dataset[dataset] = remaining_las_ids

Dataset: ppp
--------------

Number of IDs LAS: 204 (out of 514)
... of which no medication during entire study: 6
    ... of which alternative diagnosis: 1
    ... of which missing medication info: 1
... of which start medication during study: 5
    ... of which start med time unknown: 2
    ... of which start medication after week 26: 1
    ... of which alternative diagnosis during study: 0
    ... of which missing medication info: 0

Number of IDs remaining after clinical exclusions: 6

Number of IDs excluded for measurement reasons: 1
Number of IDs remaining after arm swing exclusions: 5
... Of which start with med (with start med weeks): 1  [34] 

Dataset: denovo
--------------

Number of IDs LAS: 17 (out of 99)
... of which no medication during entire study: 3
    ... of which alternative diagnosis: 0
    ... of which missing medication info: 0
... of which start medication during study: 14
    ... of which start med time unknown: 0
    ... of which start medication after week 26

In [ ]:
excluded_ids_by_category = {}
ids_remaining_after_exclusions = {}
id_valid_weeks_years_visits = {}
split_ids_range_measurement_weeks = {}

max_week_gap = 4  # Allows for surrounding weeks to be used for inter- or extrapolation

base_exclude_l1tf = {}
ids_bins = {}
for dataset in datasets:
    ids_remaining_after_exclusions[dataset] = {}

    df_clinical_dataset = df_clinical[df_clinical['dataset'] == dataset]
    start_ids = dataset_ids[dataset]

    ids_bins[dataset] = generate_ids_bins(df_clinical_dataset, dataset, start_ids, watch_side_dict[dataset], dysk_start_exclude)

    excluded_ids_by_category[dataset] = {
        'cs': {
             'clinical': {
                 'Walking aid': ids_bins[dataset]['walking_aid_visit_1'],
                f'At least {dysk_start_exclude} dyskinesia': ids_bins[dataset][f'at_least_{dysk_start_exclude}_dyskinesia_visit_1'],
                'Watch side unknown': ids_bins[dataset]['watch_side_unknown'],
                'No clinical data': ids_bins[dataset]['no_clinical_data_visit_1'],
             },
             'measurement': {}
        },
        'srm_1y': {},
        'srm_2y': {},
        'regr': {},
    }

    # Calculate exclusions for clinical data
    cs_excluded_participants = set([x for y in excluded_ids_by_category[dataset]['cs']['clinical'].values() for x in y])
    cs_ids_remaining_after_clinical_exclusion = set(start_ids) - cs_excluded_participants

    # Excluded for measurement data reasons
    ids_remaining = cs_ids_remaining_after_clinical_exclusion

    # Perform exclusion checks based on availability of measurement data (sequential)
    excluded_ids_cs_analysis_measurement = {
        'No converted data': lambda x: x not in step_ids['Week1']['converted_data'] or x not in step_ids['Week2']['converted_data'],
        'No preprocessed data': lambda x: x not in step_ids['Week1']['preprocessed_data'] or x not in step_ids['Week2']['preprocessed_data'],
        'Insufficient sensor data': lambda x: x in [x for x, y in sufficient_sensor_data[dataset][1].items() if not y] or x in [x for x, y in sufficient_sensor_data[dataset][2].items() if not y],
        'No gait predicted': lambda x: x not in step_ids['Week1']['predicted_gait_data'] or x not in step_ids['Week2']['predicted_gait_data'],
        'No arm swing predicted': lambda x: x not in digital_measures[dataset][1] or x not in digital_measures[dataset][2],
        'No arm swing predicted in vlong gait segments': lambda x: '20_inf' not in digital_measures[dataset][1].get(x, {})['filtered'] or digital_measures[dataset][1].get(x, {})['filtered']['20_inf']['duration_s'] == 0,
        'Less than 2 valid arm swing days': lambda x: valid_arm_swing_days[dataset][1][x] < min_valid_days_per_week_sufficient_sensor_data or valid_arm_swing_days[dataset][2][x] < min_valid_days_per_week_sufficient_sensor_data,
    }

    for category, condition in excluded_ids_cs_analysis_measurement.items():
        ids_excluded = [x for x in ids_remaining if condition(x)]
        ids_remaining = [x for x in ids_remaining if x not in ids_excluded]
        excluded_ids_by_category[dataset]['cs']['measurement'][category] = ids_excluded

    ids_remaining_after_exclusions[dataset]['cs'] = ids_remaining

    # LONGITUDINAL ANALYSES
    if dataset in PD_DATASETS:
        base_exclude_l1tf[dataset] = {
            'year_1': {},
            'year_2': {},
            'two_years': {},
        }

        excluded_ids_by_category[dataset]['l1tf_y1'] = {
            'clinical': {},
            'measurement': {}
        }
        excluded_ids_by_category[dataset]['l1tf_y2'] = {
            'clinical': {},
            'measurement': {}
        }
        excluded_ids_by_category[dataset]['l1tf_2y'] = {
            'clinical': {},
            'measurement': {}
        }
        excluded_ids_by_category[dataset]['l1tf_start_med'] = {
            'clinical': {},
            'measurement': {}
        }

        for year in base_exclude_l1tf[dataset].keys():
            base_exclude_l1tf[dataset][year]['clinical'] = {
                'Alternative diagnosis': IDS_ALTERNATIVE_DIAGNOSIS,
            }
        base_exclude_l1tf[dataset]['two_years']['clinical'] = {
            'Alternative diagnosis': IDS_ALTERNATIVE_DIAGNOSIS,
        }

        # Keep a tab on the participants of whom we only use a subset of the data (between visit 1 and 2, or between visit 2 and 3)
        ids_walking_aid_visit_1_2 = list(set(ids_bins[dataset]['walking_aid_visit_1'] + ids_bins[dataset]['walking_aid_visit_2']))
        ids_walking_aid_visit_2_3 = list(set(ids_bins[dataset]['walking_aid_visit_2'] + ids_bins[dataset]['walking_aid_visit_3']))
        ids_dysk_visit_1_2 = list(set(ids_bins[dataset][f'at_least_{dysk_start_exclude}_dyskinesia_visit_1'] + ids_bins[dataset][f'at_least_{dysk_start_exclude}_dyskinesia_visit_2']))
        ids_dysk_visit_2_3 = list(set(ids_bins[dataset][f'at_least_{dysk_start_exclude}_dyskinesia_visit_2'] + ids_bins[dataset][f'at_least_{dysk_start_exclude}_dyskinesia_visit_3']))

        base_exclude_l1tf[dataset]['year_1']['clinical']['Walking aid in either visit'] = ids_walking_aid_visit_1_2
        base_exclude_l1tf[dataset]['year_2']['clinical']['Walking aid in either visit'] = ids_walking_aid_visit_2_3
        base_exclude_l1tf[dataset]['two_years']['clinical']['Walking aid in any visit'] = list(set(ids_walking_aid_visit_1_2 + ids_walking_aid_visit_2_3))
        base_exclude_l1tf[dataset]['year_1']['clinical'][f'At least {dysk_start_exclude} dyskinesia in either visit'] = ids_dysk_visit_1_2
        base_exclude_l1tf[dataset]['year_2']['clinical'][f'At least {dysk_start_exclude} dyskinesia in either visit'] = ids_dysk_visit_2_3
        base_exclude_l1tf[dataset]['two_years']['clinical'][f'At least {dysk_start_exclude} dyskinesia in any visit'] = list(set(ids_dysk_visit_1_2 + ids_dysk_visit_2_3))
        base_exclude_l1tf[dataset]['year_1']['clinical']['Watch side switched this year'] = ids_bins[dataset]['watch_side_switched_visits_1_2']
        base_exclude_l1tf[dataset]['year_2']['clinical']['Watch side switched this year'] = ids_bins[dataset]['watch_side_switched_visits_2_3']
        base_exclude_l1tf[dataset]['two_years']['clinical']['Watch side switched during study'] = list(set(ids_bins[dataset]['watch_side_switched_visits_1_2'] + ids_bins[dataset]['watch_side_switched_visits_2_3']))
        base_exclude_l1tf[dataset]['year_1']['clinical']['No clinical data'] = ids_bins[dataset]['no_clinical_data_visit_1']
        base_exclude_l1tf[dataset]['year_2']['clinical']['No clinical data'] = ids_bins[dataset]['no_clinical_data_visit_2']
        base_exclude_l1tf[dataset]['two_years']['clinical']['No clinical data'] = list(set(ids_bins[dataset]['no_clinical_data_visit_1'] + ids_bins[dataset]['no_clinical_data_visit_2']))
        base_exclude_l1tf[dataset]['year_1']['clinical']['Watch side unknown'] = ids_bins[dataset]['watch_side_unknown']
        base_exclude_l1tf[dataset]['year_2']['clinical']['Watch side unknown'] = ids_bins[dataset]['watch_side_unknown']
        base_exclude_l1tf[dataset]['two_years']['clinical']['Watch side unknown'] = ids_bins[dataset]['watch_side_unknown']
        base_exclude_l1tf[dataset]['year_1']['clinical']['Start medication week unknown'] = [id for id in IDS_START_MED_TIME_UNKNOWN if id in dataset_ids[dataset]]
        base_exclude_l1tf[dataset]['year_2']['clinical']['Start medication week unknown'] = [id for id in IDS_START_MED_TIME_UNKNOWN if id in dataset_ids[dataset]]
        base_exclude_l1tf[dataset]['two_years']['clinical']['Start medication week unknown'] = [id for id in IDS_START_MED_TIME_UNKNOWN if id in dataset_ids[dataset]]

        excluded_ids_by_category[dataset]['l1tf_2y']['clinical'], l1tf_remaining_ids_after_clinical_exclusion_two_years = determine_excluded_ids_med_split(
            dataset_ids[dataset], med_info_ids, dataset, base_exclude_l1tf[dataset]['two_years']['clinical']
        )

        # Measurement
        base_exclude_l1tf[dataset]['year_1']['measurement'] = {}
        base_exclude_l1tf[dataset]['year_2']['measurement'] = {}
        base_exclude_l1tf[dataset]['two_years']['measurement'] = {}
        
        for excl_set in base_exclude_l1tf[dataset]: 
            base_exclude_l1tf[dataset][excl_set]['measurement'] = {
                'No converted data in any week': [],
                f'Insufficient preprocessed data in less than {min_valid_weeks_l1tf} valid weeks': [],
                'No valid arm swing data in any week': [],
                f'Insufficient arm swing data in less than {min_valid_weeks_l1tf} valid weeks': []
            }

        # First we check for two years, afterwards for one year
        two_year_range = range(2 - max_week_gap, 100 + max_week_gap + 1)

        for id in l1tf_remaining_ids_after_clinical_exclusion_two_years:
            has_sensor_data = any(id in sufficient_sensor_data[dataset][week] for week in two_year_range if week in sufficient_sensor_data[dataset])
            if not has_sensor_data:
                base_exclude_l1tf[dataset]['two_years']['measurement']['No converted data in any week'].append(id)
                continue

            has_sufficient_sensor_data = sum([
                sufficient_sensor_data[dataset][week][id] for week in two_year_range
                if week in sufficient_sensor_data[dataset] and id in sufficient_sensor_data[dataset][week]
            ]) >= min_valid_weeks_l1tf
            if not has_sufficient_sensor_data:
                base_exclude_l1tf[dataset]['two_years']['measurement'][f'Insufficient preprocessed data in less than {min_valid_weeks_l1tf} valid weeks'].append(id)
                continue

            has_arm_swing_data = any(id in valid_arm_swing_days[dataset][week] for week in two_year_range if week in valid_arm_swing_days[dataset])
            if not has_arm_swing_data:
                base_exclude_l1tf[dataset]['two_years']['measurement']['No valid arm swing data in any week'].append(id)
                continue

            has_sufficient_arm_swing_data = sum([
                valid_arm_swing_days[dataset][week][id] for week in two_year_range 
                if week in valid_arm_swing_days[dataset] and id in valid_arm_swing_days[dataset][week] and valid_arm_swing_days[dataset][week][id] >= min_valid_days_per_week_arm_swing 
            ]) >= min_valid_weeks_l1tf
            if not has_sufficient_arm_swing_data:
                base_exclude_l1tf[dataset]['two_years']['measurement'][f'Insufficient arm swing data in less than {min_valid_weeks_l1tf} valid weeks'].append(id)
                continue

        excluded_ids_by_category[dataset]['l1tf_2y']['measurement'], l1tf_remaining_ids_after_measurement_exclusion_two_years = determine_excluded_ids_med_split(
            l1tf_remaining_ids_after_clinical_exclusion_two_years, med_info_ids, dataset, base_exclude_l1tf[dataset]['two_years']['measurement']
        )

        ids_remaining_after_exclusions[dataset]['l1tf_2y'] = l1tf_remaining_ids_after_measurement_exclusion_two_years

        # Now check for the ones dropped whether we can include a single year of them
        # Start with dropped after measurement exclusion of two years because those dropped after
        # measurement exclusion will also be dropped if split per year
        invalid_ids_two_years = [id for id in dataset_ids[dataset] if id not in l1tf_remaining_ids_after_clinical_exclusion_two_years]
        excluded_ids_by_category[dataset]['l1tf_y1']['clinical'], l1tf_remaining_ids_after_clinical_exclusion_year_1 = determine_excluded_ids_med_split(
            invalid_ids_two_years, med_info_ids, dataset, base_exclude_l1tf[dataset]['year_1']['clinical']
        )     

        excluded_ids_by_category[dataset]['l1tf_y2']['clinical'], l1tf_remaining_ids_after_clinical_exclusion_year_2 = determine_excluded_ids_med_split(
            invalid_ids_two_years, med_info_ids, dataset, base_exclude_l1tf[dataset]['year_2']['clinical']
        )      

        l1tf_remaining_ids_in_one_year = set(l1tf_remaining_ids_after_clinical_exclusion_year_1 + l1tf_remaining_ids_after_clinical_exclusion_year_2)  

        ids_l1tf_range_weeks = {}
        for id in l1tf_remaining_ids_in_one_year:
            if id in l1tf_remaining_ids_after_clinical_exclusion_year_1 and id in l1tf_remaining_ids_after_clinical_exclusion_year_2:
                id_year = 3
                ids_l1tf_range_weeks[id] = np.arange(start_week_y1 - max_week_gap, end_week_y2 + max_week_gap + 1, 2)
            elif id in l1tf_remaining_ids_after_clinical_exclusion_year_1:
                id_year = 1
                ids_l1tf_range_weeks[id] = np.arange(start_week_y1 - max_week_gap, end_week_y1 + max_week_gap + 1, 2)
                split_ids_range_measurement_weeks[id] = ids_l1tf_range_weeks[id]
            elif id in l1tf_remaining_ids_after_clinical_exclusion_year_2:
                id_year = 2
                ids_l1tf_range_weeks[id] = np.arange(start_week_y2 - max_week_gap, end_week_y2 + max_week_gap + 1, 2)
                split_ids_range_measurement_weeks[id] = ids_l1tf_range_weeks[id]
            else:
                raise ValueError(f'ID {id} not found in either year 1 or year 2 exclusions.')

            has_sensor_data = any(id in sufficient_sensor_data[dataset][week] for week in ids_l1tf_range_weeks[id] if week in sufficient_sensor_data[dataset])
            if not has_sensor_data:
                base_exclude_l1tf[dataset][f'year_{id_year}']['measurement']['No converted data in any week'].append(id)
                continue

            has_sufficient_sensor_data = sum([
                sufficient_sensor_data[dataset][week][id] for week in ids_l1tf_range_weeks[id] 
                if week in sufficient_sensor_data[dataset] and id in sufficient_sensor_data[dataset][week]
            ]) >= min_valid_weeks_l1tf
            if not has_sufficient_sensor_data:
                base_exclude_l1tf[dataset][f'year_{id_year}']['measurement'][f'Insufficient preprocessed data in less than {min_valid_weeks_l1tf} valid weeks'].append(id)
                continue

            has_arm_swing_data = any(id in valid_arm_swing_days[dataset][week] for week in ids_l1tf_range_weeks[id] if week in valid_arm_swing_days[dataset])
            if not has_arm_swing_data:
                base_exclude_l1tf[dataset][f'year_{id_year}']['measurement']['No valid arm swing data in any week'].append(id)
                continue

            has_sufficient_arm_swing_data = sum([
                valid_arm_swing_days[dataset][week][id] for week in ids_l1tf_range_weeks[id] 
                if week in valid_arm_swing_days[dataset] and id in valid_arm_swing_days[dataset][week] and valid_arm_swing_days[dataset][week][id] >= min_valid_days_per_week_sufficient_sensor_data
            ]) >= min_valid_weeks_l1tf
            if not has_sufficient_arm_swing_data:
                base_exclude_l1tf[dataset][f'year_{id_year}']['measurement'][f'Insufficient arm swing data in less than {min_valid_weeks_l1tf} valid weeks'].append(id)
                continue


        excluded_ids_by_category[dataset]['l1tf_y1']['measurement'], l1tf_remaining_ids_after_measurement_exclusion_year_1 = determine_excluded_ids_med_split(
            l1tf_remaining_ids_after_clinical_exclusion_year_1, med_info_ids, dataset, base_exclude_l1tf[dataset]['year_1']['measurement']
        )

        excluded_ids_by_category[dataset]['l1tf_y2']['measurement'], l1tf_remaining_ids_after_measurement_exclusion_year_2 = determine_excluded_ids_med_split(
            l1tf_remaining_ids_after_clinical_exclusion_year_2, med_info_ids, dataset, base_exclude_l1tf[dataset]['year_2']['measurement']
        )

        ids_remaining_after_exclusions[dataset]['l1tf_y1'] = l1tf_remaining_ids_after_measurement_exclusion_year_1
        ids_remaining_after_exclusions[dataset]['l1tf_y2'] = l1tf_remaining_ids_after_measurement_exclusion_year_2

        valid_years_per_id_l1tf = {}
        for id in l1tf_remaining_ids_after_measurement_exclusion_year_1:
            valid_years_per_id_l1tf[id] = [1]
        for id in l1tf_remaining_ids_after_measurement_exclusion_year_2:
            valid_years_per_id_l1tf[id] = [2]
        for id in l1tf_remaining_ids_after_measurement_exclusion_two_years:
            valid_years_per_id_l1tf[id] = [1, 2]

        # Do the same for participants who start medication during the study
        all_ids_start_med = [x for x in start_ids if x in med_info_ids['groups']['start_med'][dataset]]
        ids_bins[f'{dataset}_start_med'] = generate_ids_bins(df_clinical_dataset, dataset, all_ids_start_med, watch_side_dict[dataset], dysk_start_exclude)

        med_info_ids['groups']['med'][dataset] = med_info_ids['groups']['med'][dataset] + [f'{id}_med' for id in med_info_ids['groups']['start_med'][dataset]]
        med_info_ids['groups']['no_med'][dataset] = med_info_ids['groups']['no_med'][dataset] + [f'{id}_no_med' for id in med_info_ids['groups']['start_med'][dataset]]

        # add 'med' and 'no_med' to ids_bins and remove the plain one
        for category, cat_ids in ids_bins[f'{dataset}_start_med'].items():
            start_med_cat_ids = [x for x in cat_ids if x in all_ids_start_med]

            for id in start_med_cat_ids:
                ids_bins[f'{dataset}_start_med'][category] += [f'{id}_no_med', f'{id}_med']
                ids_bins[f'{dataset}_start_med'][category] = [x for x in ids_bins[f'{dataset}_start_med'][category] if x != id]
        
        base_exclude_l1tf[dataset]['start_med'] = {
            'clinical': {},
            'measurement': {}
        }

        for med_state in ['pre_med', 'post_med']:
            base_exclude_l1tf[dataset]['start_med']['clinical'][med_state] = {
                'Start medication week unknown': [f'{id}_med' for id in IDS_START_MED_TIME_UNKNOWN if id in all_ids_start_med] + [f'{id}_no_med' for id in IDS_START_MED_TIME_UNKNOWN if id in all_ids_start_med],
                'Walking aid in any visit': [],
                f'At least {dysk_start_exclude} dyskinesia in any visit': [],
                'Watch side switched ': [],
            }
            base_exclude_l1tf[dataset]['start_med']['measurement'][med_state] = {
                'No converted data in any week': [],
                f'Insufficient preprocessed data in less than {min_valid_weeks_l1tf} valid weeks': [],
                'No valid arm swing data in any week': [],
                f'Insufficient arm swing data in less than {min_valid_weeks_l1tf} valid weeks': []
            }

        # Determine pre-med and post-med visit weeks to determine the clinical exclusions per medication state
        for id in [x for x in all_ids_start_med if x not in IDS_START_MED_TIME_UNKNOWN]:
            med_id = f'{id}_med'
            no_med_id = f'{id}_no_med'

            if id not in visit_weeks[dataset]:
                visit_weeks[dataset][id] = {
                    'visit1': 0,
                    'visit2': 52,
                    'visit3': None
                }
            elif 'visit2' not in visit_weeks[dataset][id] or visit_weeks[dataset][id]['visit2'] in [None, 'null']:
                visit_weeks[dataset][id]['visit2'] = 52  

            start_med_week = start_med_week_dict[dataset].loc[start_med_week_dict[dataset]['ID'] == id, 'StartWeek'].values[0]

            range_no_med_visits = range(0, start_med_week)
            range_med_visits = range(start_med_week, 200)

            split_ids_range_measurement_weeks[no_med_id] = np.arange(start_week_y1 - max_week_gap, start_med_week, 1)  # exclude start med week
            split_ids_range_measurement_weeks[med_id] = np.arange(start_med_week, end_week_y2 + max_week_gap + 1, 1)  # include start med week

            # Allocate visits to the med and non-med components
            no_med_visits = [int(v_nr[-1]) for v_nr, v_week in visit_weeks[dataset][id].items() if v_week in range_no_med_visits]
            no_med_visits += [no_med_visits[-1] + 1]  # Add the visit after the final measurement for clinical exclusion

            med_visits = [int(v_nr[-1]) for v_nr, v_week in visit_weeks[dataset][id].items() if v_week in range_med_visits]
            med_visits += [med_visits[0] - 1]  # Add the visit before the first measurement for clinical exclusion

            # Check if the participant adheres to the clinical requirements in the non-med and med parts                                                                                            
            if len(no_med_visits) == 2:
                uses_walking_aid_no_med = any([no_med_id in ids_bins[f'{dataset}_start_med'][f'walking_aid_visit_{v}'] for v in ['1','2']])
                uses_walking_aid_med = any([med_id in ids_bins[f'{dataset}_start_med'][f'walking_aid_visit_{v}'] for v in ['1', '2', '3']])

                has_dyskinesia_no_med = any([no_med_id in ids_bins[f'{dataset}_start_med'][f'at_least_{dysk_start_exclude}_dyskinesia_visit_{v}'] for v in ['1','2']])
                has_dyskinesia_med = any([med_id in ids_bins[f'{dataset}_start_med'][f'at_least_{dysk_start_exclude}_dyskinesia_visit_{v}'] for v in ['1', '2', '3']])
            elif len(no_med_visits) == 3:
                uses_walking_aid_no_med = any([no_med_id in ids_bins[f'{dataset}_start_med'][f'walking_aid_visit_{v}'] for v in ['1', '2', '3']])
                uses_walking_aid_med = any([med_id in ids_bins[f'{dataset}_start_med'][f'walking_aid_visit_{v}'] for v in ['2', '3']])

                has_dyskinesia_no_med = any([no_med_id in ids_bins[f'{dataset}_start_med'][f'at_least_{dysk_start_exclude}_dyskinesia_visit_{v}'] for v in ['1', '2', '3']])
                has_dyskinesia_med = any([med_id in ids_bins[f'{dataset}_start_med'][f'at_least_{dysk_start_exclude}_dyskinesia_visit_{v}'] for v in ['2', '3']])

            if uses_walking_aid_no_med:
                base_exclude_l1tf[dataset]['start_med']['clinical']['pre_med']['Walking aid in any visit'].append(no_med_id)
            if uses_walking_aid_med:
                base_exclude_l1tf[dataset]['start_med']['clinical']['post_med']['Walking aid in any visit'].append(med_id)
                
            if has_dyskinesia_no_med:
                base_exclude_l1tf[dataset]['start_med']['clinical']['pre_med'][f'At least {dysk_start_exclude} dyskinesia in any visit'].append(no_med_id)
            if has_dyskinesia_med:
                base_exclude_l1tf[dataset]['start_med']['clinical']['post_med'][f'At least {dysk_start_exclude} dyskinesia in any visit'].append(med_id)

            if '1' in no_med_visits and '2' in no_med_visits:
                watch_side_switched_no_med = no_med_id in ids_bins[f'{dataset}_start_med']['watch_side_switched_visits_1_2']
            elif '2' in no_med_visits and '3' in no_med_visits:
                watch_side_switched_no_med = med_id in ids_bins[f'{dataset}_start_med']['watch_side_switched_visits_2_3']
            else:
                watch_side_switched_no_med = False

            if watch_side_switched_no_med:
                base_exclude_l1tf[dataset]['start_med']['clinical']['pre_med']['Watch side switched'].append(no_med_id)

            if '1' in med_visits and '2' in med_visits:
                watch_side_switched_med = no_med_id in ids_bins[f'{dataset}_start_med']['watch_side_switched_visits_1_2']
            elif '2' in med_visits and '3' in med_visits:
                watch_side_switched_med = med_id in ids_bins[f'{dataset}_start_med']['watch_side_switched_visits_2_3']
            else:
                watch_side_switched_med = False

            if watch_side_switched_med:
                base_exclude_l1tf[dataset]['start_med']['clinical']['post_med']['Watch side switched'].append(med_id)

            # Now check measurements
            for med_state in ['pre_med', 'post_med']:
                if med_state == 'pre_med':
                    range_weeks = range_no_med_visits
                    id_med_state = no_med_id
                else:
                    range_weeks = range_med_visits
                    id_med_state = med_id

                has_sensor_data = any(id in sufficient_sensor_data[dataset][week] for week in range_weeks if week in sufficient_sensor_data[dataset])
                if not has_sensor_data:
                    base_exclude_l1tf[dataset]['start_med']['measurement'][med_state]['No converted data in any week'].append(id_med_state)
                    continue

                has_sufficient_sensor_data = sum([
                    sufficient_sensor_data[dataset][week][id] for week in range_weeks
                    if week in sufficient_sensor_data[dataset] and id in sufficient_sensor_data[dataset][week]
                ]) >= min_valid_weeks_l1tf
                if not has_sufficient_sensor_data:
                    base_exclude_l1tf[dataset]['start_med']['measurement'][med_state][f'Insufficient preprocessed data in less than {min_valid_weeks_l1tf} valid weeks'].append(id_med_state)
                    continue

                has_arm_swing_data = any(id in valid_arm_swing_days[dataset][week] for week in range_weeks if week in valid_arm_swing_days[dataset])
                if not has_arm_swing_data:
                    base_exclude_l1tf[dataset]['start_med']['measurement'][med_state]['No valid arm swing data in any week'].append(id_med_state)
                    continue

                has_sufficient_arm_swing_data = sum([
                    valid_arm_swing_days[dataset][week][id] for week in range_weeks
                    if week in valid_arm_swing_days[dataset] and id in valid_arm_swing_days[dataset][week] and valid_arm_swing_days[dataset][week][id] >= min_valid_days_per_week_sufficient_sensor_data
                ]) >= min_valid_weeks_l1tf
                if not has_sufficient_arm_swing_data:
                    base_exclude_l1tf[dataset]['start_med']['measurement'][med_state][f'Insufficient arm swing data in less than {min_valid_weeks_l1tf} valid weeks'].append(id_med_state)
                    continue

        all_ids_start_med = [f'{id}_med' for id in all_ids_start_med] + [f'{id}_no_med' for id in all_ids_start_med]

        excluded_ids_by_category[dataset]['l1tf_start_med']['clinical'], l1tf_remaining_ids_after_post_clinical_exclusion_start_med = determine_excluded_ids_med_split(
            all_ids_start_med, med_info_ids, dataset, base_exclude_l1tf[dataset]['start_med']['clinical'], start_med=True,
        )

        excluded_ids_by_category[dataset]['l1tf_start_med']['measurement'], l1tf_remaining_ids_after_measurement_exclusion_start_med = determine_excluded_ids_med_split(
            l1tf_remaining_ids_after_post_clinical_exclusion_start_med, med_info_ids, dataset, base_exclude_l1tf[dataset]['start_med']['measurement'], start_med=True
        )     

        ids_remaining_after_exclusions[dataset]['l1tf_start_med'] = l1tf_remaining_ids_after_measurement_exclusion_start_med

    else:
        base_exclude_l1tf[dataset] = {
            'clinical': {
                'Walking aid in any visit': list(set(ids_bins[dataset]['walking_aid_visit_1'] + ids_bins[dataset]['walking_aid_visit_2'])),
                f'At least {dysk_start_exclude} dyskinesia in any visit': list(set(ids_bins[dataset][f'at_least_{dysk_start_exclude}_dyskinesia_visit_1'] + ids_bins[dataset][f'at_least_{dysk_start_exclude}_dyskinesia_visit_2'])),
                'Watch side unknown': ids_bins[dataset]['watch_side_unknown'],
                'Watch side switched during study': ids_bins[dataset]['watch_side_switched'],
            },
            'measurement': {
                'No converted data in any week': [],
                f'Insufficient preprocessed data in less than {min_valid_weeks_l1tf} valid weeks': [],
                'No valid arm swing data in any week': [],
                f'Insufficient arm swing data in less than {min_valid_weeks_l1tf} valid weeks': []
            }
        }

        excluded_ids_by_category[dataset]['l1tf_y1'] = {
            'clinical': {},
            'measurement': {}
        }

        excluded_ids_by_category[dataset]['l1tf_y1']['clinical'], l1tf_remaining_ids_after_clinical_exclusion = determine_excluded_ids_med_split(
            dataset_ids[dataset], med_info_ids, dataset, base_exclude_l1tf[dataset]['clinical'], 
        )     

        for id in l1tf_remaining_ids_after_clinical_exclusion:
            has_sensor_data = any(id in sufficient_sensor_data[dataset][week] for week in sufficient_sensor_data[dataset])
            if not has_sensor_data:
                base_exclude_l1tf[dataset]['measurement']['No converted data in any week'].append(id)
                continue

            has_sufficient_sensor_data = sum([
                sufficient_sensor_data[dataset][week][id] for week  in sufficient_sensor_data[dataset] 
                if id in sufficient_sensor_data[dataset][week]
            ]) >= min_valid_weeks_l1tf
            if not has_sufficient_sensor_data:
                base_exclude_l1tf[dataset]['measurement'][f'Insufficient preprocessed data in less than {min_valid_weeks_l1tf} valid weeks'].append(id)
                continue

            has_arm_swing_data = any(id in valid_arm_swing_days[dataset][week] for week in valid_arm_swing_days[dataset])
            if not has_arm_swing_data:
                base_exclude_l1tf[dataset]['measurement']['No valid arm swing data in any week'].append(id)
                continue

            has_sufficient_arm_swing_data = sum([
                valid_arm_swing_days[dataset][week][id] for week in valid_arm_swing_days[dataset]
                if id in valid_arm_swing_days[dataset][week] and valid_arm_swing_days[dataset][week][id] >= min_valid_days_per_week_sufficient_sensor_data
            ]) >= min_valid_weeks_l1tf
            if not has_sufficient_arm_swing_data:
                base_exclude_l1tf[dataset]['measurement'][f'Insufficient arm swing data in less than {min_valid_weeks_l1tf} valid weeks'].append(id)
                continue

        excluded_ids_by_category[dataset]['l1tf_y1']['measurement'], l1tf_remaining_ids_after_measurement_exclusion = determine_excluded_ids_med_split(
            l1tf_remaining_ids_after_clinical_exclusion, med_info_ids, dataset, base_exclude_l1tf[dataset]['measurement']
        )  

        ids_remaining_after_exclusions[dataset]['l1tf'] = l1tf_remaining_ids_after_measurement_exclusion

with open(os.path.join(PATH_IDS, 'ids_bins.json'), 'w') as f:
    json.dump(ids_bins, f, indent=4)

In [ ]:
for dataset in datasets:
    df_clinical_dataset = df_clinical[df_clinical['dataset'] == dataset]
    start_ids = dataset_ids[dataset]

    if dataset in PD_DATASETS:
        base_exclude_srm = base_exclude_l1tf[dataset]['year_1'].copy()
        ids_bins = generate_ids_bins(df_clinical_dataset, dataset, start_ids, watch_side_dict[dataset], dysk_start_exclude)
        
        start_med_ids = ids_remaining_after_exclusions[dataset]['l1tf_start_med']
        med_start_df = start_med_week_dict[dataset]
        get_week = lambda id: med_start_df.loc[med_start_df['ID'] == strip_med_suffix_id(id), 'StartWeek'].values[0]

        base_exclude_srm['clinical']['No clinical data'] = list(set(ids_bins['no_clinical_data_visit_1'] + ids_bins['no_clinical_data_visit_2']))

        base_exclude_srm['clinical']['Starting medication during first year'] = [
            id for id in start_med_ids if get_week(id) < 46
        ]
        ids_start_med_late = [id for id in start_med_ids if get_week(id) >= 52 and id.endswith('_no_med')]

        def build_weeks(center): 
            return np.arange(center - max_week_gap, center + max_week_gap + 1, 2)

        week_windows = {
            1: {'start': build_weeks(start_week_y1), 'end': build_weeks(end_week_y1)},
            2: {'start': build_weeks(start_week_y2), 'end': build_weeks(end_week_y2)},
        }

        srm_y1_ids = (
            ids_remaining_after_exclusions[dataset]['l1tf_y1'] +
            ids_start_med_late +
            ids_remaining_after_exclusions[dataset]['l1tf_2y']
        )

        excluded_ids_by_category[dataset]['srm_1y']['clinical'], srm_remaining_ids_after_clinical_exclusion = determine_excluded_ids_med_split(
            dataset_ids[dataset], med_info_ids, dataset, base_exclude_srm['clinical'], 
        )     
    else:
        base_exclude_srm = base_exclude_l1tf[dataset].copy()
        ids_bins = generate_ids_bins(df_clinical_dataset, dataset, start_ids, watch_side_dict[dataset], dysk_start_exclude)

        srm_y1_ids = ids_remaining_after_exclusions[dataset]['l1tf']

        base_exclude_srm['clinical']['No clinical data'] = list(set(ids_bins['no_clinical_data_visit_1'] + ids_bins['no_clinical_data_visit_2']))

        excluded_ids_by_category[dataset]['srm_1y']['clinical'], srm_remaining_ids_after_clinical_exclusion = determine_excluded_ids_med_split(
            dataset_ids[dataset], med_info_ids, dataset, base_exclude_srm['clinical'], 
        )   

    excluded_ids_by_category[dataset]['srm_1y'] = excluded_ids_by_category[dataset]['l1tf_y1']
    excluded_ids_by_category[dataset]['srm_1y']['clinical'].update(base_exclude_srm['clinical'])
    
    insufficient_start = [id for id in srm_y1_ids if all(strip_med_suffix_id(id) not in sufficient_sensor_data[dataset].get(wk, set()) for wk in week_windows[1]['start'])]
    ids = [id for id in srm_y1_ids if id not in insufficient_start]

    insufficient_end = [id for id in ids if all(strip_med_suffix_id(id) not in sufficient_sensor_data[dataset].get(wk, set()) for wk in week_windows[1]['end'])]
    ids = [id for id in ids if id not in insufficient_end]

    no_arm_swing_start = [id for id in ids if all(strip_med_suffix_id(id) not in valid_arm_swing_days[dataset].get(wk, set()) for wk in week_windows[1]['start'])] 
    ids = [id for id in ids if id not in no_arm_swing_start]

    no_arm_swing_end = [id for id in ids if all(strip_med_suffix_id(id) not in valid_arm_swing_days[dataset].get(wk, set()) for wk in week_windows[1]['end'])] 
    ids = [id for id in ids if id not in no_arm_swing_end]

    no_valid_start = [
        id for id in ids 
        if all(strip_med_suffix_id(id) not in digital_measures[dataset].get(wk, set()) for wk in week_windows[1]['start']) 
        or all(digital_measures[dataset][wk][strip_med_suffix_id(id)]['filtered']['20_inf']['duration_s'] == 0 for wk in week_windows[1]['start'] if wk in digital_measures[dataset] and strip_med_suffix_id(id) in digital_measures[dataset][wk])
        or all(valid_arm_swing_days[dataset][wk][id] < min_valid_days_per_week_arm_swing for wk in week_windows[1]['start'] if wk in valid_arm_swing_days[dataset] and id in valid_arm_swing_days[dataset][wk])
    ]
    ids = [id for id in ids if id not in no_valid_start]

    no_valid_end = [
        id for id in ids 
        if all(strip_med_suffix_id(id) not in digital_measures[dataset].get(wk, set()) for wk in week_windows[1]['end']) 
        or all(digital_measures[dataset][wk][strip_med_suffix_id(id)]['filtered']['20_inf']['duration_s'] == 0 for wk in week_windows[1]['end'] if wk in digital_measures[dataset] and strip_med_suffix_id(id) in digital_measures[dataset][wk])
        or all(valid_arm_swing_days[dataset][wk][id] < min_valid_days_per_week_arm_swing for wk in week_windows[1]['end'] if wk in valid_arm_swing_days[dataset] and id in valid_arm_swing_days[dataset][wk])
    ]
    ids = [id for id in ids if id not in no_valid_end]

    base_exclude_srm['measurement']['year_1'] = {
        'Insufficient sensor data in first 6 weeks': sorted(insufficient_start),
        'Insufficient sensor data in last 6 weeks': sorted(insufficient_end),
        'No arm swing data in first 6 weeks': sorted(no_arm_swing_start),
        'No arm swing data in last 6 weeks': sorted(no_arm_swing_end),
        'No valid arm swing data in first 6 weeks': sorted(no_valid_start),
        'No valid arm swing data in last 6 weeks': sorted(no_valid_end),
    }

    excluded_ids_by_category[dataset]['srm_1y']['measurement'], \
    ids_remaining_after_exclusions[dataset]['srm_1y'] = determine_excluded_ids_med_split(
        srm_y1_ids, med_info_ids, dataset, base_exclude_srm['measurement']['year_1']
    )

    if dataset in PD_DATASETS:
        # ---- 2-Year SRM Filtering ----
        base_exclude_srm = base_exclude_l1tf[dataset]['two_years'].copy()
        ids_bins = generate_ids_bins(df_clinical_dataset, dataset, start_ids, watch_side_dict[dataset], dysk_start_exclude)

        start_med_ids = [id for id in ids_remaining_after_exclusions[dataset]['l1tf_start_med'] if get_week(id) < 100]

        base_exclude_srm['clinical']['No clinical data'] = list(set(ids_bins['no_clinical_data_visit_1'] + ids_bins['no_clinical_data_visit_3']))
        base_exclude_srm['clinical']['Starting medication'] = list(set([strip_med_suffix_id(id) for id in start_med_ids if get_week(id) < 100]))

        med_info_ids_modified = med_info_ids.copy()
        med_info_ids_modified['groups']['no_med'][dataset] = [strip_med_suffix_id(id) for id in med_info_ids['groups']['no_med'][dataset]]

        excluded_ids_by_category[dataset]['srm_2y']['clinical'], srm_remaining_ids_after_clinical_exclusion = determine_excluded_ids_med_split(
            dataset_ids[dataset], med_info_ids_modified, dataset, base_exclude_srm['clinical'], 
        )   

        insufficient_start = [id for id in srm_remaining_ids_after_clinical_exclusion if all(id not in sufficient_sensor_data[dataset].get(wk, set()) for wk in week_windows[1]['start'])]
        ids = [id for id in srm_remaining_ids_after_clinical_exclusion if id not in insufficient_start]

        insufficient_end = [id for id in ids if all(id not in sufficient_sensor_data[dataset].get(wk, set()) for wk in week_windows[2]['end'])]
        ids = [id for id in ids if id not in insufficient_end]

        no_arm_swing_start = [id for id in ids if all(id not in valid_arm_swing_days[dataset].get(wk, set()) for wk in week_windows[1]['start'])] 
        ids = [id for id in ids if id not in no_arm_swing_start]

        no_arm_swing_end = [id for id in ids if all(id not in valid_arm_swing_days[dataset].get(wk, set()) for wk in week_windows[2]['end'])] 
        ids = [id for id in ids if id not in no_arm_swing_end]

        # Valid start: digital measures computed of start weeks, at least 2 valid arm swing days per week
        no_valid_start = [
            id for id in ids 
            if all(id not in digital_measures[dataset].get(wk, set()) for wk in week_windows[1]['start']) 
            or all(digital_measures[dataset][wk][id]['filtered']['20_inf']['duration_s'] == 0 for wk in week_windows[1]['start'] if wk in digital_measures[dataset] and id in digital_measures[dataset][wk])
            or all(valid_arm_swing_days[dataset][wk][id] < min_valid_days_per_week_arm_swing for wk in week_windows[1]['start'] if wk in valid_arm_swing_days[dataset] and id in valid_arm_swing_days[dataset][wk])
        ]
        ids = [id for id in ids if id not in no_valid_start]

        no_valid_end = [
            id for id in ids 
            if all(id not in digital_measures[dataset].get(wk, set()) for wk in week_windows[2]['end']) 
            or all(digital_measures[dataset][wk][id]['filtered']['20_inf']['duration_s'] == 0 for wk in week_windows[2]['end'] if wk in digital_measures[dataset] and id in digital_measures[dataset][wk])
            or all(valid_arm_swing_days[dataset][wk][id] < min_valid_days_per_week_arm_swing for wk in week_windows[2]['end'] if wk in valid_arm_swing_days[dataset] and id in valid_arm_swing_days[dataset][wk])
        ]
        ids = [id for id in ids if id not in no_valid_end]   

        base_exclude_srm['measurement']['two_years'] = {
            'Insufficient sensor data in first 6 weeks': sorted(insufficient_start),
            'Insufficient sensor data in last 6 weeks': sorted(insufficient_end),
            'No arm swing data in first 6 weeks': sorted(no_arm_swing_start),
            'No arm swing data in last 6 weeks': sorted(no_arm_swing_end),
            'No valid arm swing data in first 6 weeks': sorted(no_valid_start),
            'No valid arm swing data in last 6 weeks': sorted(no_valid_end),
        }

        ids_remaining_after_exclusions[dataset]['srm_2y'] = ids

        excluded_ids_by_category[dataset]['srm_2y']['measurement'], \
            ids_remaining_after_exclusions[dataset]['srm_2y'] = determine_excluded_ids_med_split(
                srm_remaining_ids_after_clinical_exclusion, med_info_ids_modified, dataset, base_exclude_srm['measurement']['two_years']
            )
        
    # REGRESSION
    # For the regression we examine the possible effect of change in medication, dyskinesia, etc. on the outcome measures.
    # Therefore, we start with the SRM 2-year participants, and additionally exclude participants with missing LEDD
    # information and those starting medication during the study

    if dataset in PD_DATASETS:
        ids = [strip_med_suffix_id(id) for id in ids_remaining_after_exclusions[dataset]['srm_2y']]
        ids = [id for id in ids if id not in med_info_ids['groups']['start_med'][dataset] and 
               id not in med_info_ids['groups']['no_med'][dataset] and id not in IDS_LEDD_MISSING_VISIT_1 and 
               id not in IDS_LEDD_DOUBTFUL and id not in IDS_USE_ANTICHOLINERGIC_MEDS and id not in IDS_DIAGNOSIS_DATE_DOUBTFUL]
        
        ids_remaining_after_exclusions[dataset]['regr'] = ids
    
# Store ids
with open(PATH_IDS / 'ids_remaining_after_exclusions.json', 'w') as f:
    json.dump(ids_remaining_after_exclusions, f)

with open(PATH_IDS / 'excluded_ids_by_category.json', 'w') as f:
    json.dump(excluded_ids_by_category, f)

with open(PATH_IDS / 'med_info_ids.json', 'w') as f:
    json.dump(med_info_ids, f)

In [ ]:
digital_measures_pre_df = []
for dataset, dataset_vals in digital_measures.items():
    for week_nr, week_vals in dataset_vals.items():
        for subject, subject_vals in week_vals.items():

            try:
                sufficient_arm_swing = False if subject in valid_arm_swing_days[dataset][week_nr] and valid_arm_swing_days[dataset][week_nr][subject] < min_valid_days_per_week_arm_swing else True
            except KeyError:
                start_med_week = start_med_week_dict[dataset].loc[start_med_week_dict[dataset]['ID'] == subject, 'StartWeek'].values[0]
                if start_med_week > week_nr:
                    suffix = '_no_med'
                else:
                    suffix = '_med'
                sufficient_arm_swing = False if valid_arm_swing_days[dataset][week_nr][f'{subject}{suffix}'] < min_valid_days_per_week_arm_swing else True

            cross_sectional = True if sufficient_arm_swing and subject in ids_remaining_after_exclusions[dataset]['cs'] and week_nr == 2 else False
            icc = True if sufficient_arm_swing and subject in ids_remaining_after_exclusions[dataset]['cs'] and week_nr in [1, 2] else False

            if dataset in PD_DATASETS:
                l1tf_year_1 = True if (
                    sufficient_arm_swing and 
                    subject in ids_remaining_after_exclusions[dataset]['l1tf_y1'] and 
                    week_nr in split_ids_range_measurement_weeks[subject]
                ) else False
                l1tf_year_2 = True if (
                    sufficient_arm_swing and 
                    subject in ids_remaining_after_exclusions[dataset]['l1tf_y2'] and 
                    week_nr in split_ids_range_measurement_weeks[subject]
                ) else False
                l1tf_year_12 = True if (
                    sufficient_arm_swing and 
                    subject in ids_remaining_after_exclusions[dataset]['l1tf_2y'] 
                ) else False
                l1tf_pre_med = True if (
                    sufficient_arm_swing and 
                    f'{subject}_no_med' in ids_remaining_after_exclusions[dataset]['l1tf_start_med'] and 
                    len(split_ids_range_measurement_weeks[f'{subject}_no_med']) > 1 and 
                    week_nr in split_ids_range_measurement_weeks[f'{subject}_no_med']
                ) else False
            
                l1tf_post_med = True if (
                    sufficient_arm_swing and 
                    f'{subject}_med' in ids_remaining_after_exclusions[dataset]['l1tf_start_med'] and 
                    len(split_ids_range_measurement_weeks[f'{subject}_med']) > 1 and 
                    week_nr in split_ids_range_measurement_weeks[f'{subject}_med']
                ) else False

                srm_year_1 = True if (
                    sufficient_arm_swing and 
                    (
                        subject in ids_remaining_after_exclusions[dataset]['srm_1y'] or 
                        f'{subject}_med' in ids_remaining_after_exclusions[dataset]['srm_1y'] or 
                        f'{subject}_no_med' in ids_remaining_after_exclusions[dataset]['srm_1y'] 
                    )
                 ) else False
                srm_year_12 = True if (
                    sufficient_arm_swing and 
                    (
                        subject in ids_remaining_after_exclusions[dataset]['srm_2y'] or 
                        f'{subject}_med' in ids_remaining_after_exclusions[dataset]['srm_2y'] or 
                        f'{subject}_no_med' in ids_remaining_after_exclusions[dataset]['srm_2y']
                    ) 
                ) else False
                regr = True if (
                    sufficient_arm_swing and 
                    subject in ids_remaining_after_exclusions[dataset]['regr'] 
                ) else False
            else:
                l1tf_year_1 = True if sufficient_arm_swing and subject in ids_remaining_after_exclusions[dataset]['l1tf'] else False
                l1tf_year_2 = False
                l1tf_year_12 = False
                l1tf_pre_med = False
                l1tf_post_med = False

                srm_year_1 = True if sufficient_arm_swing and subject in ids_remaining_after_exclusions[dataset]['srm_1y'] else False
                srm_year_12 = False
                regr = False
            
            if dataset != 'controls':
                try:
                    aff_side = 'mas' if subject in affected_side_ids[dataset]['mas'] else ('las' if subject in affected_side_ids[dataset]['las'] else 'unknown')
                except IndexError:
                    aff_side = np.nan

            for filter_type, filter_vals in subject_vals.items():
                for segment_category, category_vals in filter_vals.items():
                    if segment_category != 'minutes_of_data':
                        row = {
                            'dataset': dataset,
                            'week': week_nr,
                            'id': subject,
                            'filter_type': filter_type,
                            'segment_category': segment_category,
                            'include_in_cross_sectional_analysis': cross_sectional,
                            'include_in_icc_analysis': icc,
                            'include_in_l1tf_y1_analysis': l1tf_year_1,
                            'include_in_l1tf_y2_analysis': l1tf_year_2,
                            'include_in_l1tf_y12_analysis': l1tf_year_12,
                            'include_in_l1tf_pre_med_analysis': l1tf_pre_med,
                            'include_in_l1tf_post_med_analysis': l1tf_post_med,
                            'include_in_srm_y1_analysis': srm_year_1,
                            'include_in_srm_y12_analysis': srm_year_12,
                            'include_in_regression_analysis': regr,
                            'sufficient_arm_swing_this_week': sufficient_arm_swing
                        }

                        if dataset != 'controls': 
                            row['affected_side'] = aff_side

                        row.update(category_vals) 

                        digital_measures_pre_df.append(row)

# Convert to DataFrame
df_digital_measures = pd.DataFrame(digital_measures_pre_df)
df_digital_measures.loc[df_digital_measures['dataset'].isin(['ppp', 'denovo']), 'population'] = 'pd'
df_digital_measures.loc[df_digital_measures['dataset']=='controls', 'population'] = 'controls'

no_med_ids = med_info_ids['groups']['no_med']['ppp'] + med_info_ids['groups']['no_med']['denovo']
med_ids = med_info_ids['groups']['med']['ppp'] + med_info_ids['groups']['med']['denovo']

df_digital_measures.loc[df_digital_measures['id'].isin(med_ids), 'med_state'] = 'med'
df_digital_measures.loc[df_digital_measures['id'].isin(no_med_ids), 'med_state'] = 'no_med'
df_digital_measures.loc[df_digital_measures['population'] == 'controls', 'med_state'] = 'controls'

for dataset in [ds for ds in datasets if ds in PD_DATASETS]:
    for subject in ids_remaining_after_exclusions[dataset]['l1tf_start_med']:
        start_med_week = start_med_week_dict[dataset].loc[start_med_week_dict[dataset]['ID'] == strip_med_suffix_id(subject), 'StartWeek'].values[0]
        df_digital_measures.loc[
            (df_digital_measures['id']==strip_med_suffix_id(subject)) & 
            (df_digital_measures['week'] < start_med_week), 
            'med_state'
        ] = 'no_med'
        df_digital_measures.loc[
            (df_digital_measures['id']==strip_med_suffix_id(subject)) & 
            (df_digital_measures['week'] >= start_med_week), 
            'med_state'
        ] = 'med'

os.makedirs(BASE_PATH / 'measures', exist_ok=True)

df_digital_measures.to_parquet(BASE_PATH / 'measures' / 'digital_measures.parquet', index=False)

In [ ]:
aggregates = ['median', '95p', 'mean_cov', 'median_cov', 'mean_std', 'median_std']
quantifications = ['range_of_motion']

measures = [f'{agg}_{quant}' for agg in aggregates for quant in quantifications]

for filter_type in ['filtered', 'unfiltered']:
    for segment_category in df_digital_measures['segment_category'].unique():
        os.makedirs(BASE_PATH / 'measures' / f'{filter_type}_gait' / segment_category, exist_ok=True)

        # Store subset for l1 trend filter in Matlab
        df_l1tf = df_digital_measures.loc[
            (df_digital_measures['filter_type']==filter_type) &
            (df_digital_measures['week'] != 1) &
            (df_digital_measures['sufficient_arm_swing_this_week']) &
            (df_digital_measures['segment_category'] == segment_category) &
            (
                df_digital_measures['include_in_l1tf_y1_analysis'] |
                df_digital_measures['include_in_l1tf_y2_analysis'] |
                df_digital_measures['include_in_l1tf_y12_analysis'] |
                df_digital_measures['include_in_l1tf_pre_med_analysis'] |
                df_digital_measures['include_in_l1tf_post_med_analysis']
            )
        ]

        if sum(pd.isna(df_l1tf['med_state'])) > 0:
            raise ValueError('med_state column contains NaN values.')
        
        for population in ['pd_med', 'pd_no_med', 'controls']:
            if population == 'pd_med':
                df_l1tf_population = df_l1tf.loc[df_l1tf['med_state'] == 'med']
            elif population == 'pd_no_med':
                df_l1tf_population = df_l1tf.loc[df_l1tf['med_state'] == 'no_med']
            else:
                df_l1tf_population = df_l1tf.loc[df_l1tf['med_state'] == 'controls']

            for measure in measures:
                df_pivot = df_l1tf_population.pivot(index="id", columns="week", values=measure).reset_index(drop=False).rename_axis(None, axis=1)

                ids = df_pivot['id'].values.tolist()

                df_pivot = df_pivot.drop(columns=['id']).apply(pd.to_numeric, errors='coerce')

                df_pivot.to_csv(BASE_PATH / 'measures' / f'{filter_type}_gait' / segment_category / f'{population}_measure_{measure}.csv', index=False)

            with open(BASE_PATH / 'measures' / f'{filter_type}_gait' / segment_category / f'{population}_ids.txt', 'w') as f:
                f.write('\n'.join(ids))